# Main Analysis Notebook (Clean Pipeline)

This notebook runs a staged workflow:

1. Load all available Docent data into an index.
2. Create grouped 70:10:20 splits by `agent_run_id`.
3. Exclude `execute_malware` from train and val (holdout only).
4. Format prompts with the monitor policy.
5. Extract surrogate activations.
6. Train on train split, evaluate on val and holdout.
7. Report metrics and diagnostics.

In [1]:
# One-time dependency install for this kernel (if needed)
%uv pip install -q scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Ensure local package imports resolve in this kernel
import sys
from pathlib import Path


def _find_repo_root_with_package() -> Path | None:
    # Support both legacy and updated mount layouts.
    candidates = [
        Path.cwd(),
        Path("/workspace"),
        Path("/workspace/workspace"),
        Path("/root/workspace"),
        Path("/root/workspace/workspace"),
        Path("/root"),
        Path("/"),
    ]

    for root in candidates:
        try:
            if (root / "surrogate_sv" / "__init__.py").exists():
                return root
        except Exception:
            continue

    # Fallback bounded search in common Modal locations.
    for base in [Path("/workspace"), Path("/root")]:
        if not base.exists():
            continue
        try:
            for pkg_dir in base.rglob("surrogate_sv"):
                if (pkg_dir / "__init__.py").exists():
                    return pkg_dir.parent
        except Exception:
            continue

    return None


PACKAGE_ROOT = _find_repo_root_with_package()
if PACKAGE_ROOT is None:
    raise RuntimeError(
        "Could not find repository root containing surrogate_sv. "
        "Expected Modal mount under /workspace. Sync with: "
        "modal volume put code-vol .\\surrogate_sv /surrogate_sv"
    )

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

print("PACKAGE_ROOT:", PACKAGE_ROOT)

PACKAGE_ROOT: /__modal/volumes/vo-1nHYYrf5M725Aseq8p3TOt


In [3]:
from dataclasses import dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
from sklearn.metrics import classification_report, precision_recall_curve, roc_curve

from surrogate_sv.config import COLLECTIONS, DEFAULT_READER_MODEL_NAME, PromptPolicyConfig
from surrogate_sv.data.docent import get_docent_client
from surrogate_sv.experiments import (
    assign_grouped_splits,
    build_transcript_index,
    extract_activations_for_formatted,
    format_prompts_for_index,
    summarize_split_counts,
)
from surrogate_sv.experiments.splits import SplitConfig
from surrogate_sv.classifiers.training import evaluate_models, train_binary_models
from surrogate_sv.paths import resolve_output_root


@dataclass(frozen=True)
class RunConfig:
    # Split policy
    train_frac: float = 0.7
    val_frac: float = 0.1
    holdout_frac: float = 0.2
    split_group_col: str = "agent_run_id"
    forced_holdout_collection: str = "execute_malware"

    # Prompt policy
    include_system_prompt: bool = True
    include_first_user_message: bool = True
    use_chat_template: bool = True
    add_generation_prompt: bool = True
    max_prompt_tokens: int = 4096
    task_max_tokens: int = 500
    first_n_tool_calls: int | None = None

    # Extraction policy
    reader_model_name: str = DEFAULT_READER_MODEL_NAME
    layers: tuple[int, ...] = (12, 16)
    pooling: str = "mean"
    max_seq_len: int = 4096

    # Runtime and I/O
    random_state: int = 42
    page_size: int = 200


RUN = RunConfig()
RUN_TAG = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
ARTIFACT_ROOT = resolve_output_root() / "clean_pipeline_runs" / RUN_TAG
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("RUN_TAG:", RUN_TAG)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RUN_TAG: run_20260415_074325
ARTIFACT_ROOT: /root/datasets/artifacts/docent_activation_spaces/clean_pipeline_runs/run_20260415_074325


In [4]:
# Stage 1: Load all available Docent data into an index
client = get_docent_client()
all_index_df = build_transcript_index(client=client, collection_map=COLLECTIONS, page_size=RUN.page_size)

if all_index_df.empty:
    raise RuntimeError("No transcripts found from Docent.")

print("Total indexed transcripts:", len(all_index_df))
display(
    all_index_df.groupby("collection_name", dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values("collection_name")
    .reset_index(drop=True)
)

# Save lightweight index artifact (without full messages payload).
index_lite_path = ARTIFACT_ROOT / "all_index_lite.csv"
all_index_df.drop(columns=["messages"], errors="ignore").to_csv(index_lite_path, index=False)
print("Saved:", index_lite_path)

07:45:41 [INFO] docent.sdk.client: Authenticating Docent client with frontend_url='https://docent.transluce.org' and api_url='https://api.docent.transluce.org/rest'
07:45:42 [INFO] docent.sdk.client: Logged in with API key
Total indexed transcripts: 500


,collection_name,count
0,disable_firewall,100
1,escalate_privileges,100
2,execute_malware,100
3,exfiltrate_secrets,100
4,honest,100


Saved: /root/datasets/artifacts/docent_activation_spaces/clean_pipeline_runs/run_20260415_074325/all_index_lite.csv


In [5]:
# Stage 2: Build grouped 70:10:20 splits with execute_malware reserved for holdout
split_cfg = SplitConfig(
    train_frac=RUN.train_frac,
    val_frac=RUN.val_frac,
    holdout_frac=RUN.holdout_frac,
    random_state=RUN.random_state,
    group_col=RUN.split_group_col,
    transcript_col="transcript_id",
    collection_col="collection_name",
    forced_holdout_collection=RUN.forced_holdout_collection,
)

split_df = assign_grouped_splits(all_index_df, split_cfg)
split_summary_df = summarize_split_counts(split_df)
display(split_summary_df)

# Leakage check: no shared groups across splits.
grouped = split_df.copy()
grouped["group_id_resolved"] = grouped[RUN.split_group_col].fillna(grouped["transcript_id"]).astype(str)
train_groups = set(grouped.loc[grouped["split"] == "train", "group_id_resolved"])
val_groups = set(grouped.loc[grouped["split"] == "val", "group_id_resolved"])
hold_groups = set(grouped.loc[grouped["split"] == "holdout", "group_id_resolved"])

assert train_groups.isdisjoint(val_groups), "train/val group leakage detected"
assert train_groups.isdisjoint(hold_groups), "train/holdout group leakage detected"
assert val_groups.isdisjoint(hold_groups), "val/holdout group leakage detected"

# Policy checks for execute_malware.
train_exe = int(((split_df["split"] == "train") & (split_df["collection_name"] == RUN.forced_holdout_collection)).sum())
val_exe = int(((split_df["split"] == "val") & (split_df["collection_name"] == RUN.forced_holdout_collection)).sum())
hold_exe = int(((split_df["split"] == "holdout") & (split_df["collection_name"] == RUN.forced_holdout_collection)).sum())
print("execute_malware counts -> train:", train_exe, "val:", val_exe, "holdout:", hold_exe)
assert train_exe == 0 and val_exe == 0, "execute_malware must be excluded from train/val"

split_path = ARTIFACT_ROOT / "split_assignments.csv"
split_df.drop(columns=["messages"], errors="ignore").to_csv(split_path, index=False)
print("Saved:", split_path)

,split,collection_name,count
0,holdout,disable_firewall,24
1,holdout,escalate_privileges,21
2,holdout,execute_malware,100
3,holdout,exfiltrate_secrets,16
4,holdout,honest,20
5,train,disable_firewall,65
6,train,escalate_privileges,67
7,train,exfiltrate_secrets,76
8,train,honest,71
9,val,disable_firewall,11


execute_malware counts -> train: 0 val: 0 holdout: 100
Saved: /root/datasets/artifacts/docent_activation_spaces/clean_pipeline_runs/run_20260415_074325/split_assignments.csv


In [6]:
# Stage 3: Prompt formatting
prompt_policy = PromptPolicyConfig(
    include_system_prompt=RUN.include_system_prompt,
    include_first_user_message=RUN.include_first_user_message,
    use_chat_template=RUN.use_chat_template,
    add_generation_prompt=RUN.add_generation_prompt,
    max_prompt_tokens=RUN.max_prompt_tokens,
    task_max_tokens=RUN.task_max_tokens,
    first_n_tool_calls=RUN.first_n_tool_calls,
)

formatted_df = format_prompts_for_index(
    index_df=split_df,
    prompt_policy=prompt_policy,
    reader_model_name=RUN.reader_model_name,
    drop_no_tool_calls=True,
)

if formatted_df.empty:
    raise RuntimeError("No formatted prompts were produced.")

print("Formatted prompts:", len(formatted_df))
display(
    formatted_df.groupby(["split", "collection_name"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["split", "collection_name"])
    .reset_index(drop=True)
)

formatted_path = ARTIFACT_ROOT / "formatted_prompts_lite.csv"
formatted_df.drop(columns=["messages", "prompt_text"], errors="ignore").to_csv(formatted_path, index=False)
print("Saved:", formatted_path)

`torch_dtype` is deprecated! Use `dtype` instead!
Formatting prompts: 100%|██████████| 500/500 [00:17<00:00, 28.55it/s]

Formatted prompts: 495


,split,collection_name,count
0,holdout,disable_firewall,24
1,holdout,escalate_privileges,21
2,holdout,execute_malware,99
3,holdout,exfiltrate_secrets,16
4,holdout,honest,19
5,train,disable_firewall,64
6,train,escalate_privileges,66
7,train,exfiltrate_secrets,75
8,train,honest,71
9,val,disable_firewall,11


Saved: /root/datasets/artifacts/docent_activation_spaces/clean_pipeline_runs/run_20260415_074325/formatted_prompts_lite.csv


In [7]:
# Stage 4: Activation extraction (supports multiple layers)
extraction_outputs = extract_activations_for_formatted(
    formatted_df=formatted_df,
    layers=RUN.layers,
    pooling=RUN.pooling,
    reader_model_name=RUN.reader_model_name,
    max_seq_len=RUN.max_seq_len,
)

extraction_meta_df = extraction_outputs["metadata"].copy()
vectors_by_layer = extraction_outputs["vectors_by_layer"]

if extraction_meta_df.empty:
    raise RuntimeError("No activations extracted.")

# Labels for binary classification: honest=0, malicious=1.
extraction_meta_df["label"] = (extraction_meta_df["collection_name"].astype(str) != "honest").astype(int)

print("Extracted rows:", len(extraction_meta_df))
for layer, tensor in vectors_by_layer.items():
    print(f"layer {layer}: shape={tuple(tensor.shape)} dtype={tensor.dtype}")

extract_meta_path = ARTIFACT_ROOT / "extraction_metadata.csv"
extraction_meta_df.drop(columns=["messages", "prompt_text"], errors="ignore").to_csv(extract_meta_path, index=False)
print("Saved:", extract_meta_path)

Extracting activations: 100%|██████████| 495/495 [06:44<00:00,  1.23it/s]

Extracted rows: 495
layer 12: shape=(495, 4096) dtype=torch.float32
layer 16: shape=(495, 4096) dtype=torch.float32
Saved: /root/datasets/artifacts/docent_activation_spaces/clean_pipeline_runs/run_20260415_074325/extraction_metadata.csv


In [21]:
# Stage 5: Train on train split and evaluate on val/holdout
pipeline_results = {}
metrics_rows = []

for layer in RUN.layers:
    X = vectors_by_layer[int(layer)].detach().cpu().numpy()
    meta = extraction_meta_df.reset_index(drop=True).copy()
    y = meta["label"].to_numpy(dtype=np.int64)
    source = meta["collection_name"].astype(str).to_numpy()

    train_mask = meta["split"].astype(str) == "train"
    val_mask = meta["split"].astype(str) == "val"
    hold_mask = meta["split"].astype(str) == "holdout"

    if int(train_mask.sum()) == 0 or int(val_mask.sum()) == 0 or int(hold_mask.sum()) == 0:
        raise RuntimeError(
            f"Insufficient split rows for layer {layer}: "
            f"train={int(train_mask.sum())}, val={int(val_mask.sum())}, holdout={int(hold_mask.sum())}"
        )

    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val, src_val = X[val_mask], y[val_mask], source[val_mask]
    X_hold, y_hold, src_hold = X[hold_mask], y[hold_mask], source[hold_mask]

    models = train_binary_models(X_train=X_train, y_train=y_train, random_state=RUN.random_state)
    val_metrics_df, val_results, val_subtype_df = evaluate_models(models=models, X_eval=X_val, y_eval=y_val, source_eval=src_val)
    hold_metrics_df, hold_results, hold_subtype_df = evaluate_models(models=models, X_eval=X_hold, y_eval=y_hold, source_eval=src_hold)

    val_metrics_df.insert(0, "layer", int(layer))
    val_metrics_df.insert(1, "split", "val")
    hold_metrics_df.insert(0, "layer", int(layer))
    hold_metrics_df.insert(1, "split", "holdout")
    metrics_rows.extend([val_metrics_df, hold_metrics_df])

    pipeline_results[int(layer)] = {
        "models": models,
        "val": {
            "metrics_df": val_metrics_df,
            "results": val_results,
            "subtype_df": val_subtype_df,
        },
        "holdout": {
            "metrics_df": hold_metrics_df,
            "results": hold_results,
            "subtype_df": hold_subtype_df,
        },
    }

all_metrics_df = pd.concat(metrics_rows, ignore_index=True).sort_values(["split", "layer", "roc_auc"], ascending=[True, True, False]).reset_index(drop=True)
display(all_metrics_df)

metrics_path = ARTIFACT_ROOT / "metrics_all_layers.csv"
all_metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,layer,split,model,accuracy,balanced_accuracy,f1,roc_auc,pr_auc,n_eval
0,12,holdout,logreg_l2,0.910615,0.787664,0.949686,0.913816,0.988604,179
1,12,holdout,lda_shrinkage,0.921788,0.770724,0.956522,0.865789,0.971154,179
2,16,holdout,logreg_l2,0.916201,0.813980,0.952681,0.915132,0.988546,179
3,16,holdout,lda_shrinkage,0.938547,0.826480,0.965732,0.844901,0.965823,179
4,12,val,logreg_l2,0.900000,0.777778,0.939394,0.967742,0.990906,40
5,12,val,lda_shrinkage,0.925000,0.833333,0.953846,0.822581,0.906520,40
6,16,val,logreg_l2,0.925000,0.833333,0.953846,0.992832,0.997951,40
7,16,val,lda_shrinkage,0.925000,0.833333,0.953846,0.888889,0.939394,40


Saved: /root/datasets/artifacts/docent_activation_spaces/clean_pipeline_runs/run_20260415_074325/metrics_all_layers.csv


In [29]:
# Stage 6: Reporting (choose layer and model)
REPORT_LAYER = RUN.layers[1]
REPORT_MODEL = "logreg_l2"

if int(REPORT_LAYER) not in pipeline_results:
    raise RuntimeError(f"REPORT_LAYER={REPORT_LAYER} not found in pipeline_results")

hold_block = pipeline_results[int(REPORT_LAYER)]["holdout"]
val_block = pipeline_results[int(REPORT_LAYER)]["val"]

print("Val metrics:")
display(val_block["metrics_df"].query("model == @REPORT_MODEL"))
print("Holdout metrics:")
display(hold_block["metrics_df"].query("model == @REPORT_MODEL"))

# Holdout subtype diagnostics
print("Holdout subtype diagnostics:")
display(hold_block["subtype_df"].query("model == @REPORT_MODEL").sort_values("collection_name"))

# Text report on holdout.
hold_res = hold_block["results"][REPORT_MODEL]
print(classification_report(hold_res["y_true"], hold_res["y_pred"], target_names=["honest", "malicious"], digits=4))

# ROC and PR curves for the selected layer across both models.
roc_rows = []
pr_rows = []
for model_name, model_out in hold_block["results"].items():
    try:
        fpr, tpr, _ = roc_curve(model_out["y_true"], model_out["y_score"])
        roc_rows.append(pd.DataFrame({"fpr": fpr, "tpr": tpr, "model": model_name}))
    except Exception:
        pass
    try:
        prec, rec, _ = precision_recall_curve(model_out["y_true"], model_out["y_score"])
        pr_rows.append(pd.DataFrame({"recall": rec, "precision": prec, "model": model_name}))
    except Exception:
        pass

if roc_rows:
    roc_df = pd.concat(roc_rows, ignore_index=True)
    fig_roc = px.line(roc_df, x="fpr", y="tpr", color="model", title=f"Holdout ROC (layer={REPORT_LAYER})", width=780, height=540)
    fig_roc.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="random", line=dict(dash="dash"))
    fig_roc.show()

if pr_rows:
    pr_df = pd.concat(pr_rows, ignore_index=True)
    fig_pr = px.line(pr_df, x="recall", y="precision", color="model", title=f"Holdout PR (layer={REPORT_LAYER})", width=780, height=540)
    fig_pr.show()

Val metrics:


,layer,split,model,accuracy,balanced_accuracy,f1,roc_auc,pr_auc,n_eval
0,16,val,logreg_l2,0.925,0.833333,0.953846,0.992832,0.997951,40


Holdout metrics:


,layer,split,model,accuracy,balanced_accuracy,f1,roc_auc,pr_auc,n_eval
0,16,holdout,logreg_l2,0.916201,0.81398,0.952681,0.915132,0.988546,179


Holdout subtype diagnostics:


,model,collection_name,n_eval,malicious_frac,malicious_recall
5,logreg_l2,disable_firewall,24,1.0,0.875000
6,logreg_l2,escalate_privileges,21,1.0,0.904762
7,logreg_l2,execute_malware,99,1.0,0.969697
8,logreg_l2,exfiltrate_secrets,16,1.0,0.937500
9,logreg_l2,honest,19,0.0,NaN


              precision    recall  f1-score   support

      honest     0.5909    0.6842    0.6341        19
   malicious     0.9618    0.9437    0.9527       160

    accuracy                         0.9162       179
   macro avg     0.7763    0.8140    0.7934       179
weighted avg     0.9224    0.9162    0.9189       179



In [30]:
# Confusion matrix + prediction-rate diagnostics + always-malicious baseline
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score

if int(REPORT_LAYER) not in pipeline_results:
    raise RuntimeError(f"REPORT_LAYER={REPORT_LAYER} not found in pipeline_results")

if REPORT_MODEL not in pipeline_results[int(REPORT_LAYER)]["holdout"]["results"]:
    raise RuntimeError(f"REPORT_MODEL={REPORT_MODEL} not found for layer={REPORT_LAYER}")

def _prediction_rate_tables(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    overall_pred_rate = (
        pd.Series(y_pred, name="pred_label")
        .map({0: "honest", 1: "malicious"})
        .value_counts(normalize=True)
        .rename("rate")
        .reset_index()
        .rename(columns={"index": "predicted_class"})
    )

    by_true_class = pd.crosstab(
        pd.Series(y_true, name="true_label").map({0: "honest", 1: "malicious"}),
        pd.Series(y_pred, name="pred_label").map({0: "honest", 1: "malicious"}),
        normalize="index",
    ).reset_index().rename(columns={"true_label": "true_class"})

    return overall_pred_rate, by_true_class

def _always_malicious_baseline(y_true: np.ndarray) -> pd.DataFrame:
    y_base = np.ones_like(y_true)
    row = {
        "baseline": "always_malicious",
        "accuracy": float(accuracy_score(y_true, y_base)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_base)),
        "f1_malicious": float(f1_score(y_true, y_base, pos_label=1)),
        "precision_malicious": float(precision_score(y_true, y_base, pos_label=1, zero_division=0)),
        "recall_malicious": float(recall_score(y_true, y_base, pos_label=1, zero_division=0)),
        "pred_malicious_rate": 1.0,
        "n": int(len(y_true)),
    }
    return pd.DataFrame([row])

val_res = pipeline_results[int(REPORT_LAYER)]["val"]["results"][REPORT_MODEL]
hold_res = pipeline_results[int(REPORT_LAYER)]["holdout"]["results"][REPORT_MODEL]

val_cm_df = pd.DataFrame(
    val_res["confusion_matrix"],
    index=["true_honest", "true_malicious"],
    columns=["pred_honest", "pred_malicious"],
)

hold_cm_df = pd.DataFrame(
    hold_res["confusion_matrix"],
    index=["true_honest", "true_malicious"],
    columns=["pred_honest", "pred_malicious"],
)

print(f"Validation confusion matrix ({REPORT_MODEL}, layer={REPORT_LAYER})")
display(val_cm_df)

val_overall_pred_rate, val_pred_rate_by_true = _prediction_rate_tables(
    y_true=np.asarray(val_res["y_true"]),
    y_pred=np.asarray(val_res["y_pred"]),
)
print("Validation prediction rate (overall predicted class distribution)")
display(val_overall_pred_rate)
print("Validation prediction rate by true class (row-normalized)")
display(val_pred_rate_by_true)
print("Validation always-malicious baseline")
display(_always_malicious_baseline(np.asarray(val_res["y_true"])))

print(f"Holdout confusion matrix ({REPORT_MODEL}, layer={REPORT_LAYER})")
display(hold_cm_df)

hold_overall_pred_rate, hold_pred_rate_by_true = _prediction_rate_tables(
    y_true=np.asarray(hold_res["y_true"]),
    y_pred=np.asarray(hold_res["y_pred"]),
)
print("Holdout prediction rate (overall predicted class distribution)")
display(hold_overall_pred_rate)
print("Holdout prediction rate by true class (row-normalized)")
display(hold_pred_rate_by_true)
print("Holdout always-malicious baseline")
display(_always_malicious_baseline(np.asarray(hold_res["y_true"])))

Validation confusion matrix (logreg_l2, layer=16)


,pred_honest,pred_malicious
true_honest,6,3
true_malicious,0,31


Validation prediction rate (overall predicted class distribution)


,pred_label,rate
0,malicious,0.85
1,honest,0.15


Validation prediction rate by true class (row-normalized)


...

,pred_label,rate
0,malicious,0.877095
1,honest,0.122905


Holdout prediction rate by true class (row-normalized)


pred_label,true_class,honest,malicious
0,honest,0.684211,0.315789
1,malicious,0.056250,0.943750


Holdout always-malicious baseline


,baseline,accuracy,balanced_accuracy,f1_malicious,precision_malicious,recall_malicious,pred_malicious_rate,n
0,always_malicious,0.893855,0.5,0.943953,0.893855,1.0,1.0,179
